# Weed Detection - GPU training on Google Colab

Trains **YOLO11** on the `Francesco/weed-crop-aerial` dataset using Colab's free GPU
(~50-100x faster than CPU). Evaluates on the test split and lets you download
`best.pt` + metrics.

**Steps:**
1. `Runtime` -> `Change runtime type` -> Hardware accelerator = **T4 GPU** -> Save
2. `Runtime` -> `Run all`
3. **Wait for cell 4 (training) to fully finish** before the later cells will work
   (Run all does this for you; only an issue if you click cells by hand).

If Colab disconnects, just `Run all` again - it is safe to re-run every cell.

In [ ]:
# 1. Check the GPU is on
import torch
assert torch.cuda.is_available(), 'No GPU! Runtime > Change runtime type > T4 GPU, then Run all again.'
print('GPU:', torch.cuda.get_device_name(0))

In [ ]:
# 2. Get the project code + install deps (Colab already has a matching torch).
#    Safe to re-run: re-clones only if the folder is missing.
import os
if not os.path.isdir('/content/repo'):
    !git clone --depth 1 https://github.com/Bhawna109/AI-based-Weed-detection.git /content/repo
%cd /content/repo
!pip -q install ultralytics datasets
import ultralytics; ultralytics.checks()

In [ ]:
# 3. Download + convert the dataset to YOLO format (writes configs/data.yaml).
%cd /content/repo
if not os.path.isdir('dataset/images/train'):
    !python src/fetch_hf_dataset.py --dataset Francesco/weed-crop-aerial
!python src/verify_dataset.py --samples 0

In [ ]:
# 4. TRAIN.  This is the long cell - ~30-60 min for 100 epochs on a T4.
#    Wait for it to print a final metrics table before running the cells below.
#    Change --model to yolo11n.pt (faster) or yolo11m.pt (more accurate).
%cd /content/repo
!python src/train.py \
    --model yolo11s.pt \
    --epochs 100 \
    --batch 32 \
    --imgsz 640 \
    --device 0 \
    --workers 2 \
    --cache ram \
    --patience 25 \
    --name weed_yolo11s_colab

In [ ]:
# 5. Evaluate on the untouched test split -> real Precision / Recall / mAP.
%cd /content/repo
BEST = 'results/runs/weed_yolo11s_colab/weights/best.pt'
assert os.path.exists(BEST), (
    'best.pt not found - cell 4 (training) has not finished yet. '
    'Wait for it to complete, then run this cell again.')
!python src/evaluate.py --weights $BEST --split test --name test_eval_colab

In [ ]:
# 6. Run predictions on the test images and show a few.
%cd /content/repo
BEST = 'results/runs/weed_yolo11s_colab/weights/best.pt'
assert os.path.exists(BEST), 'Run cell 4 (training) to completion first.'
!python src/predict.py --weights $BEST --source dataset/images/test --conf 0.25 --name colab_test

import glob, random
from IPython.display import Image, display
imgs = glob.glob('results/predictions/colab_test/*.jpg')
for p in random.sample(imgs, min(6, len(imgs))):
    display(Image(filename=p, width=500))

In [ ]:
# 7. Training curves + confusion matrix.
%cd /content/repo
!python src/plot_results.py --run results/runs/weed_yolo11s_colab
from IPython.display import Image, display
for p in ['results/training_curves.png',
          'results/test_confusion_matrix_normalized.png',
          'results/test_PR_curve.png']:
    try: display(Image(filename=p, width=650))
    except Exception as e: print('not found yet:', p)

In [ ]:
# 8. Package weights + metrics + plots and download to your computer.
%cd /content/repo
import shutil, os
BEST = 'results/runs/weed_yolo11s_colab/weights/best.pt'
assert os.path.exists(BEST), 'Run cell 4 (training) to completion first.'
os.makedirs('download', exist_ok=True)
shutil.copy(BEST, 'download/best.pt')
for f in ['metrics_test.json', 'training_curves.png',
          'test_PR_curve.png', 'test_confusion_matrix_normalized.png']:
    if os.path.exists(f'results/{f}'):
        shutil.copy(f'results/{f}', f'download/{f}')
shutil.make_archive('weed_yolo11s_colab', 'zip', 'download')
print('zip size:', round(os.path.getsize('weed_yolo11s_colab.zip')/1e6, 1), 'MB')
from google.colab import files
files.download('weed_yolo11s_colab.zip')

## After download

On your PC, unzip and place the files:

- `best.pt` -> `results/runs/weed_yolo11s_colab/weights/best.pt` (create the folders)
- the `.png` / `.json` files -> `results/`

Then update the Results table in `README.md` with the new test numbers and commit.
Run local predictions any time with:

```bash
python src/predict.py --weights results/runs/weed_yolo11s_colab/weights/best.pt --source <image_or_folder>
```